# Question 0

Kaplan-Meier estimate

Rausand (2004) pp 476.

## prompt

```py
def km_estimate(ttf_data: list[float], censored_data: list[float], t: float) -> float:
    ''' Given a list of time-to-failure data ttf_data, and a list of censoring time in censored_data, calculate the Kaplan-Meier estimate of reliability function at time t. 
    Example,
    >>> km_estimate([31.7, 39.2, 57.5, 65.8, 70.0, 105.8, 110.0], [65.0, 75.0, 75.2, 87.5, 88.3, 94.2, 101.7, 109.2, 130.0], 58)
    0.813
    '''
```

## unit test

In [ ]:
def check(candidate):
    import numpy as np
    
    tol = 1e-3

    # Test.
    ttf_data_1 = [22.5, 34.7, 40.1, 78.2, 79.3]
    censored_data_1 = [54.3, 62.5, 85.4, 90.8]
    t_1 = 52 # test regular time.
    t_2 = 34.7 # test an exact match on time from failure data.
    t_3 = 91 # test after censoring and failure upper bound.
    t_4 = 22 # test before censoring and failure lower bound.

    try:
        # Test 1
        assert abs(candidate(ttf_data_1, censored_data_1, t_1) - 0.6666666) < tol
        assert abs(candidate(ttf_data_1, censored_data_1, t_2) - 0.7777777777) < tol
        assert abs(candidate(ttf_data_1, censored_data_1, t_3) - 0.33333333) < tol
        assert abs(candidate(ttf_data_1, censored_data_1, t_4) - 1) < tol

        ttf_data_2 = [22.5, 40.1, 34.7, 78.2, 79.3] # test non-sorted failure 
        assert abs(candidate(ttf_data_2, censored_data_1, t_1) - 0.66666666) < tol

        return 'pass'
    except:
        return 'fail'

# Question 1

Likelihood function of right-censoring, left-censoring and interval-censoring data

## prompt

```py
def loglikelihood(pdf: callable, theta: list, data: list[float], right_censored: list[float], left_censored: list[float], interval_censor: list[list[float, float]]) -> float:
    ''' Given a callable `pdf`, which represents the probability density function of a lifetime distribution and take as inputs parameters t (time) and theta (a list of model parameters), a list `theta`, which represents the parameter values of pdf, a list `data` representing $n$ complete time-to-failure data, a list `right_censored`, representing the censoring time of $n_rc$ right-censored data, a list `left_censored`, representing the censoring time of $n_lc$ left-censored data, and a list `interval_censor` whose elements are list [ll, ul], representing the lower and upper bounds of the interval censoring data, calculate the value of the loglikelihood function of given `data`, `right_censored`, `left_censored` at the parameter value `theta`.
    Example:
    >>> loglikelihood(lambda t, theta: 1/theta*np.exp(-t / theta), 10, [10], [5], [10], [[5, 10]])
    >>> -5.6940123679
    '''
```



## unit test

In [ ]:
def check(candidate):
    import numpy as np
    from scipy.stats import weibull_min
    
    tol = 1e-3

    try:
        # Test 1
        pdf = lambda t, theta: 1/theta*np.exp(-t / theta)
        theta = 10

        assert abs(candidate(pdf, theta, [10], [5], [10], [[5, 10]]) + 5.6940123679) < tol
        # Test 2
        def pdf(t, theta):
            eta = theta[0]
            beta = theta[1]

            return weibull_min.pdf(t, beta, scale=eta)
        theta = [100, 1]

        data = [10, 15]
        right_censored = [5, 6]
        left_censored = [1, 2]
        interval_censor = [[5, 10], [3, 5]]

        assert abs(candidate(pdf, theta, data, right_censored, left_censored, interval_censor) + 25.1251471) < tol

        return 'pass'
    except:
        return 'fail'

# Question 2

Likelihood function of right-censoring, left-censoring and interval-censoring data

## prompt

```py
def loglikelihood(reliability: callable, theta: list, data: list[float], right_censored: list[float], left_censored: list[float], interval_censor: list[list[float, float]]) -> float:
    ''' Given a callable `reliability`, which represents the reliability function of a lifetime distribution and take as inputs parameters t (time) and theta (a list of model parameters), a list `theta`, which represents the parameter values of pdf, a list `data` representing $n$ complete time-to-failure data, a list `right_censored`, representing the censoring time of $n_rc$ right-censored data, a list `left_censored`, representing the censoring time of $n_lc$ left-censored data, and a list `interval_censor` whose elements are list [ll, ul], representing the lower and upper bounds of the interval censoring data, calculate the value of the loglikelihood function of given `data`, `right_censored`, `left_censored` at the parameter value `theta`.
    Example:
    >>> loglikelihood(lambda t, theta: np.exp(-t / theta), 10, [10], [5], [10], [[5, 10]])
    >>> -5.6940123679
    '''
```



## Unit test

In [ ]:
def check(candidate):
    import numpy as np
    from scipy.stats import weibull_min
    
    tol = 1e-3

    try:
        # Test 1
        assert abs(candidate(lambda t, theta: np.exp(-t / theta), 10, [10], [5], [10], [[5, 10]]) + 5.6940123679) < tol
        
        # Test 2
        def reliability(t, theta):
            eta = theta[0]
            beta = theta[1]

            return 1-weibull_min.cdf(t, beta, scale=eta)
        theta = [100, 1]

        data = [10, 15]
        right_censored = [5, 6]
        left_censored = [1, 2]
        interval_censor = [[5, 10], [3, 5]]

        assert abs(candidate(reliability, theta, data, right_censored, left_censored, interval_censor) + 25.1251471) < tol

        return 'pass'
    except:
        return 'fail'

# Question 3 

MLE with censoring - Generic case

## prompt

```py
def mle_censored(density: callable, initial_guess: list[float], data: list[float], right_censored: list[float]=None, left_censored: list[float]=None, interval_censored: list[list[float, float]]=None) -> list:
    ''' Given a density function `density` with inputs t (time) and theta (parameter vector), a list of initial guess of the parameter theta, a list of complete time-to-failure data in `data`, a list of right censored data in `right_censored`, and a list of left censored data in `left_censored`, and a list of interval cenored data (interval_censor), which is in the format of [[lower, upper]], return the MLE of the parameter theta of the density function. 
    Example:
    >>> mle_censored(lambda t,_lambda: _lambda * np.exp(-_lambda * t), 
    >>>                   [.01], [1.2, 2.2, 4.1, 5.6, 8.4, 16.2, 21.7, 41, 49.3, 60.5, 94, 98],
    >>>                   [1.8, 11.8, 13.4, 29, 42, 42.4, 61, 99.2])
    >>> 0.01707468
    '''
```

## unit tests

In [ ]:
def check(candidate):
    import numpy as np
    from scipy.stats import weibull_min
    
    tol = 1e-3

    try:
        # Test 1
        assert abs(candidate(lambda t,_lambda: _lambda * np.exp(-_lambda * t), 
                       [.01], [1.2, 2.2, 4.1, 5.6, 8.4, 16.2, 21.7, 41, 49.3, 60.5, 94, 98],
                       [1.8, 11.8, 13.4, 29, 42, 42.4, 61, 99.2])[0] - 0.01707468) < tol
        # Test 2
        pdf = lambda t, theta: 1/theta*np.exp(-t / theta)
        assert abs(candidate(pdf, [.1], [10, 5, 3, 5, 9], [15, 16], [12], [[10, 11]])[0] - 11.20241411) < tol

        # Test 3
        def pdf(t, theta):
            eta = theta[0]
            beta = theta[1]

            return weibull_min.pdf(t, beta, scale=eta)
        
        initial_guess = [100, 1]
        data = [58,75,36,52,63,65,22,17,28,64,23,40,73,45,52,36,52,60,13,55,82,55,34,57,23,42,66,35,34,25]
        assert abs(candidate(pdf, initial_guess, data)[0] - 51.85858775) < tol
        assert abs(candidate(pdf, initial_guess, data)[1] - 2.80074465) < tol

        return 'pass'
    except:
        return 'fail'

# Question 4

Bayesian reliability analysis - Failure rate

Rausand (2004) pp 543

## prompt

```py
def failure_rate_updating(alpha: float, beta: float, observed_failure_time: list[float]) -> float:
    ''' Assume that the lifetime follows an exponential distribution and the prior distribution of failure rate is a gamma distribution with parameters alpha and beta. Given alpha, beta and a list of observed failure times in `observed_failure_time`, return the posterior mean of the failure rate.
    >>> Examples: failure_rate_updating(2, 1, [10, 20])
    >>> 0.129032
    '''
```

## unit tests

In [ ]:
def check(candidate):
    import numpy as np
    from scipy.stats import weibull_min
    
    tol = 1e-3

    try:
        assert abs(candidate(2, 1, [10, 20])-4/31)<tol
        # Test 1
        assert abs(candidate(10, 2, [100, 200, 300, 400])-0.0139720)<tol

        return 'pass'
    except:
        return 'fail'

# Question 5 

Degradation model - General path model

## prompt

```py
def gpm(t_measure: float, data: float, y_th: float, t_pred: float) -> float:
    ''' Assume that the degradation follows a general path model with a linear degradation model and a white noise. Given a list of `t_measure` and a list of floats `data`, representing the measurement time and measured degradation, respectively; a float `y_th` reprsenting the failure threshold (when the degradation is larger than `y_th`, the system fails). Calculate the probability of failure at a future moment `t_pred`.
    >>> Examples: gpm([10, 20, 30, 40, 50, 60], [9.8, 12.5, 13.6, 12.8, 15.6, 14.2], 40, 340)
    >>> 0.36501126
    '''
```


## Unit test

In [ ]:
def check(candidate):    
    tol = 1e-3

    try:
        assert abs(candidate([10, 20, 30, 40, 50, 60], [9.8, 12.5, 13.6, 12.8, 15.6, 14.2], 40, 340)-0.365011260)<tol
        
        return 'pass'
    except:
        return 'fail'